# LangChain: RAG & Q&A

## Outline
* مفهوم RAG
* ساخت knowledge base (loader → split → embed → store)
* 2-Step RAG با LCEL
* Agentic RAG با `create_agent`
* مقایسه دو روش


In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


In [26]:
# pip install langchain-openai langchain-community faiss-cpu

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


## ۱. لود PDF



In [29]:
# ── Step 1: Load (تغییر یافته برای PDF) ──

#pip install pypdf
from langchain_community.document_loaders import PyPDFLoader

# لود کردن فایل PDF
loader = PyPDFLoader("capital-market-qa.pdf")
docs = loader.load()

print(f"Loaded {len(docs)} pages from PDF")
# نمایش بخشی از صفحه اول برای تست
print(f"Sample content: {docs[0].page_content[:100]}...")

Loaded 3 pages from PDF
Sample content: مهمترین سوالات درباره بازار سرمایه
خلاصهای آموزشی از پرتکرارترین پرسشهای بازار بورس و اوراق بهادار
م...


## ۲. ساخت Knowledge Base

In [32]:
# ── Step 2: Split ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
splits = splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")


Split into 5 chunks


In [34]:
# ── Step 3: Embed & Store ──
embeddings = OllamaEmbeddings(model="bge-m3")


In [42]:
# ساخت vector store
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created!")

# تست similarity search
query = "ساعات کاری معاملات از کی است؟"
results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content}")
    print("*"*20)


Vector store created!

Top 2 results for 'ساعات کاری معاملات از کی است؟':
  1. مهمترین سوالات درباره بازار سرمایه
خلاصهای آموزشی از پرتکرارترین پرسشهای بازار بورس و اوراق بهادار
منبع اصلی
۱. کلیات بازار سرمایه
س: بازار سرمایه به چه بخشهایی تقسیم میشود؟
ج: بازار بورس و اوراق بهادار برای تأمین مالی شرکتها و ایجاد بستری شفاف برای سرمایهگذاری شکل گرفته و
بهطور کلی به دو بخش اصلی تقسیم میشود: بازار اولیه که در آن سهام شرکتها برای نخستین بار عرضه
میشود، و بازار ثانویه که در آن خرید و فروش سهام میان سرمایهگذاران انجام میشود.
س: چگونه میتوان سرمایهگذاری در بورس را آغاز کرد؟
ج: پس از تکمیل ثبتنام در سامانه سجام و دریافت کد بورسی، سرمایهگذار میتواند از طریق سامانههای
معاملاتی کارگزاریها معاملات خود را شروع کند.
۲. زمان و نحوه معاملات
س: ساعت معاملات بازار سرمایه چه زمانی است؟
ج: ثبت سفارش خرید و فروش سهام بورسی و فرابورسی از شنبه تا چهارشنبه (بهجز تعطیلات رسمی) و از
ساعت ۵۴:۸ تا ۰۳:۲۱ انجام میشود. بین ساعت ۵۴:۸ تا ۰۰:۹ معاملهای انجام نمیشود و نخستین معامله
رأس ساعت ۹ صورت میگیرد.
س: پیشگشایش یعن

## ۳. 2-Step RAG با LCEL

In [44]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-5.2", model_provider="openai", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.
If you don't know the answer, say "I don't have that information."

Context:
{context}"""),
("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2-Step RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# تست
response = rag_chain.invoke("کی از متقاضیان ثبت نام انجام میگیرد؟")
print(response)


I don't have that information.


In [46]:
response = rag_chain.invoke("پیش گشایش یعنی چی؟")
print(response)


پیشگشایش به بازه زمانی **۸:۵۴ تا ۹:۰۰** گفته می‌شود که در آن فعالان بازار فقط می‌توانند سفارش‌های خود را **ثبت، ویرایش یا حذف** کنند و **معامله‌ای انجام نمی‌شود**.  
در **۵ دقیقه پایانی** این بازه (**۸:۵۵ تا ۹:۰۰**) امکان **ویرایش یا حذف سفارش** هم وجود ندارد.


In [48]:
# تست 2
response = rag_chain.invoke("چه روزهایی میشود معامله انجام داد؟")
print(response)

ثبت سفارش خرید و فروش سهام بورسی و فرابورسی از **شنبه تا چهارشنبه** (به‌جز **تعطیلات رسمی**) انجام می‌شود.
